# FUR Anchor Trajectory Geometry

Variance test + correlation test for the geometry-faithfulness hypothesis.

**Input**: `data/fur_anchor_embeddings.pkl`  
Each record has:
- `anchor_embeddings`: dict of 9 numpy arrays (the trajectory points, middle layer)
- `ff_soft`: float — P(correct) before unlearning (faithfulness proxy)
- `condition`, `epoch`: which unlearning run produced the state

**Tests**:
1. **Variance test** — do trajectory geometry metrics vary meaningfully across questions?
2. **Correlation test** — do those metrics correlate with `ff_soft`?

Geometric feature functions and PCA projection are adapted directly from `geometry_compare.ipynb`.

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
from sklearn.decomposition import PCA

PKL_PATH = 'data/fur_anchor_embeddings.pkl'

with open(PKL_PATH, 'rb') as f:
    records = pickle.load(f)

print(f'Loaded {len(records)} record(s)')
for r in records:
    anc = r['anchor_embeddings']
    found = sum(1 for v in anc.values() if v is not None)
    print(f"  {r['question_id']:25s}  ff_soft={r['ff_soft']:.4f}  "
          f"condition={r['condition']}  epoch={r['epoch']}  anchors={found}/9")

## 1. PCA projection — 9 anchor points → 3D

In [ ]:
TRAJ_KEYS = [
    'Answer_A', 'Reasoning_A', 'Answer_B', 'Reasoning_B',
    'Answer_C', 'Reasoning_C', 'Answer_D', 'Reasoning_D', 'Final_Answer',
]

def project_trajectory(record, n_components=3):
    """
    Project the 9 anchor embeddings for one question to n_components via PCA.
    Returns a DataFrame with columns [anchor, pc1, pc2, pc3] and the PCA object.
    Returns None if fewer than 3 valid anchors.
    """
    anc = record['anchor_embeddings']
    vecs   = [anc.get(k) for k in TRAJ_KEYS]
    valid  = [(k, v) for k, v in zip(TRAJ_KEYS, vecs) if v is not None]
    if len(valid) < 3:
        return None, None
    keys, mats = zip(*valid)
    X = np.stack(mats).astype(np.float32)
    nc = min(n_components, len(valid))
    pca = PCA(n_components=nc, random_state=42).fit(X)
    proj = pca.transform(X)
    # Pad to 3 columns if fewer components are available
    while proj.shape[1] < 3:
        proj = np.hstack([proj, np.zeros((proj.shape[0], 1))])
    df = pd.DataFrame(proj, columns=['pc1', 'pc2', 'pc3'])
    df.insert(0, 'anchor', list(keys))
    return df, pca

# Project all records
projections = []
for r in records:
    df, pca = project_trajectory(r)
    if df is not None:
        ev = pca.explained_variance_ratio_
        print(f"{r['question_id']:25s}  "
              f"PC1={ev[0]:.3f}  PC2={ev[1]:.3f}  "
              f"ff_soft={r['ff_soft']:.4f}")
    projections.append({'record': r, 'df': df, 'pca': pca})

## 2. Geometric feature functions

Copied directly from `geometry_compare.ipynb` — no changes needed.

In [ ]:
def pts(df): return df[['pc1','pc2','pc3']].values

def arclength(df):
    p = pts(df); return np.sum(np.linalg.norm(np.diff(p, axis=0), axis=1))

def net_displacement(df):
    p = pts(df); return np.linalg.norm(p[-1] - p[0])

def straightness(df):
    a, n = arclength(df), net_displacement(df); return n / a if a > 1e-9 else 0.

def mean_step_size(df):
    p = pts(df); return np.linalg.norm(np.diff(p, axis=0), axis=1).mean()

def step_size_std(df):
    p = pts(df); return np.linalg.norm(np.diff(p, axis=0), axis=1).std()

def mean_curvature(df):
    p = pts(df)
    if len(p) < 3: return 0.
    d1 = np.diff(p, axis=0)
    d2 = np.diff(d1, axis=0)
    norms = np.linalg.norm(d1[:-1], axis=1)
    cross = np.linalg.norm(np.cross(d1[:-1], d2), axis=1)
    mask  = norms > 1e-9
    return float(np.mean(cross[mask] / (norms[mask]**2 + 1e-9))) if mask.any() else 0.

def final_pc1(df): return float(pts(df)[-1, 0])
def final_pc2(df): return float(pts(df)[-1, 1])
def final_pc3(df): return float(pts(df)[-1, 2])
def pc1_range(df): return float(pts(df)[:, 0].max() - pts(df)[:, 0].min())
def pc2_range(df): return float(pts(df)[:, 1].max() - pts(df)[:, 1].min())

METRICS = [
    ('arclength',        arclength),
    ('net_displacement', net_displacement),
    ('straightness',     straightness),
    ('mean_step_size',   mean_step_size),
    ('step_size_std',    step_size_std),
    ('mean_curvature',   mean_curvature),
    ('final_pc1',        final_pc1),
    ('final_pc2',        final_pc2),
    ('final_pc3',        final_pc3),
    ('pc1_range',        pc1_range),
    ('pc2_range',        pc2_range),
]

## 3. Compute metric table

In [ ]:
rows = []
for p in projections:
    if p['df'] is None:
        continue
    r   = p['record']
    df  = p['df']
    row = {
        'question_id': r['question_id'],
        'condition':   r['condition'],
        'epoch':       r['epoch'],
        'ff_soft':     r['ff_soft'],
    }
    for name, fn in METRICS:
        row[name] = round(fn(df), 6)
    rows.append(row)

df_metrics = pd.DataFrame(rows).set_index('question_id')
display(df_metrics)

## 4. Test 1 — Variance test

Do trajectory geometry metrics vary meaningfully across questions?  
Reports coefficient of variation (CV = std/mean) per metric.  
A CV > 0.2 suggests the metric carries discriminative signal.

In [ ]:
metric_names = [m[0] for m in METRICS]

var_rows = []
for name in metric_names:
    vals = df_metrics[name].values.astype(float)
    mean = vals.mean()
    std  = vals.std()
    cv   = std / abs(mean) if abs(mean) > 1e-12 else float('nan')
    flag = '**' if cv > 0.2 else ('*' if cv > 0.1 else '')
    var_rows.append({'Metric': name, 'Mean': round(mean, 4),
                     'Std': round(std, 4), 'CV': round(cv, 3), '': flag})

df_var = pd.DataFrame(var_rows)
print('Variance test (CV > 0.2 = high variability **)')
display(df_var.style
    .apply(lambda r: ['background-color:#d4edda' if r['CV'] > 0.2 else
                       'background-color:#fff3cd' if r['CV'] > 0.1 else ''
                       for _ in r], axis=1)
    .format({'Mean': '{:.4f}', 'Std': '{:.4f}', 'CV': '{:.3f}'})
    .hide(axis='index'))

## 5. Test 2 — Correlation with FF-SOFT

Spearman rank correlation between each geometry metric and `ff_soft`.  
With n ≤ 6 the threshold for p < 0.05 (two-tailed) is |ρ| ≥ ~0.83.

In [ ]:
ff_vals = df_metrics['ff_soft'].values.astype(float)

corr_rows = []
for name in metric_names:
    metric_vals = df_metrics[name].values.astype(float)
    if len(metric_vals) < 4:
        corr_rows.append({'Metric': name, 'Spearman rho': float('nan'),
                          'p-value': float('nan'), 'Sig': '(n<4)'})
        continue
    rho, p = stats.spearmanr(metric_vals, ff_vals)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    corr_rows.append({'Metric': name, 'Spearman rho': round(rho, 3),
                      'p-value': p, 'Sig': sig})

df_corr = pd.DataFrame(corr_rows)
print('Correlation test: geometry metric vs. FF-SOFT')
display(df_corr.style
    .apply(lambda r: ['background-color:#fff3cd' if r['p-value'] < 0.05 else ''
                       for _ in r], axis=1)
    .format({'Spearman rho': '{:.3f}', 'p-value': '{:.3e}'})
    .hide(axis='index'))

## 6. Trajectory visualisation — colored by FF-SOFT

In [ ]:
valid_projs = [p for p in projections if p['df'] is not None]
if not valid_projs:
    print('No valid projections to plot.')
else:
    ff_all   = np.array([p['record']['ff_soft'] for p in valid_projs])
    cmap     = cm.RdYlGn           # red=low ff_soft (unfaithful), green=high (faithful)
    norm     = plt.Normalize(ff_all.min(), ff_all.max())

    fig = plt.figure(figsize=(16, 6))

    # ── 3D ──────────────────────────────────────────────────────────────────
    ax3d = fig.add_subplot(121, projection='3d')
    for proj in valid_projs:
        r   = proj['record']
        p   = pts(proj['df'])
        c   = cmap(norm(r['ff_soft']))
        lbl = f"{r['question_id']} ({r['ff_soft']:.3f})"
        ax3d.plot(p[:,0], p[:,1], p[:,2], '-o', color=c, lw=2,
                  markersize=5, label=lbl, alpha=0.9)
        ax3d.scatter(*p[0],  s=80, c='black', marker='s', zorder=5)
        ax3d.scatter(*p[-1], s=80, color=[c], marker='*', zorder=5)
    ax3d.set_xlabel('PC1'); ax3d.set_ylabel('PC2'); ax3d.set_zlabel('PC3')
    ax3d.set_title('3D Trajectories  (■=start  ★=end)')
    ax3d.legend(fontsize=7)

    # ── PC1 vs PC2 ──────────────────────────────────────────────────────────
    ax2d = fig.add_subplot(122)
    for idx, proj in enumerate(valid_projs):
        r   = proj['record']
        p   = pts(proj['df'])
        c   = cmap(norm(r['ff_soft']))
        lbl = f"{r['question_id']} ({r['ff_soft']:.3f})"
        ax2d.plot(p[:,0], p[:,1], '-o', color=c, lw=2, markersize=7,
                  label=lbl, alpha=0.9)
        for i, (x, y) in enumerate(zip(p[:,0], p[:,1])):
            ax2d.annotate(str(i+1), (x, y), fontsize=7, ha='center',
                          va='bottom', color=c)
    ax2d.set_xlabel('PC1'); ax2d.set_ylabel('PC2')
    ax2d.set_title('PC1 vs PC2  (step numbers labelled)')
    ax2d.legend(fontsize=7)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax2d, label='FF-SOFT')

    plt.suptitle('FUR Anchor Trajectories — colored by FF-SOFT', fontsize=13)
    plt.tight_layout()
    plt.show()

## 7. Per-question metric breakdown

In [ ]:
display(
    df_metrics[['ff_soft'] + metric_names]
    .sort_values('ff_soft')
    .style
    .background_gradient(subset=['ff_soft'], cmap='RdYlGn')
    .format({col: '{:.4f}' for col in ['ff_soft'] + metric_names})
)